# 01 - Levine--Tristram signature functions

`gaknot` represents a signature function by exact discontinuity data. This notebook explains that sparse representation before using it in torus-knot, satellite, and connected-sum calculations.

## Learning objectives

- interpret a jump weight and the midpoint convention;
- evaluate the function periodically on `R/Z`;
- add, negate, shift, and rescale sparse signature functions;
- recognize the satellite formula in an iterated-torus-knot example; and
- render a signature as a step function.

## 1. Imports and a first signature

In [ ]:
from pathlib import Path
import sys

repository_root = Path.cwd()
if repository_root.name == "notebooks":
    repository_root = repository_root.parent
source_directory = repository_root / "src"
if str(source_directory) not in sys.path:
    sys.path.insert(0, str(source_directory))

import matplotlib.pyplot as plt
from sage.all import QQ
from gaknot import (
    GeneralizedAlgebraicKnot,
    SignatureFunction,
    SignaturePloter,
)

trefoil = GeneralizedAlgebraicKnot.torus_knot(2, 3)
trefoil_signature = trefoil.signature()
trefoil_signature.jumps_counter

For the positive trefoil, the stored weights are `-1` at `1/6` and `+1` at `5/6`. Crossing a weight `j` changes the signature by `2j`. The total weight is zero, as expected for a periodic function after making one full circuit of the unit circle.

In [ ]:
print("sum of jump weights:", trefoil_signature.total_sign_jump())
print("first largest one-sided value:", trefoil_signature.extremum())

## 2. Midpoint values at discontinuities

At a jump location `x` of weight `j`, the package uses

`sigma(x) = 2 * (sum of weights strictly before x) + j`.

Consequently, the stored value is halfway between the left- and right-hand limits. The exact rational offset below makes the three cases visible without floating-point rounding.

In [ ]:
jump = QQ(1) / 6
epsilon = QQ(1) / 1000

{
    "left of 1/6": trefoil_signature(jump - epsilon),
    "at 1/6": trefoil_signature(jump),
    "right of 1/6": trefoil_signature(jump + epsilon),
}

The midpoint convention matters whenever an invariant is evaluated exactly at a root of the Alexander polynomial. It also explains why reading only a plotted horizontal segment does not determine the value at its endpoint.

## 3. Periodicity and exact arguments

Evaluation reduces every argument modulo one. Therefore arguments differing by an integer describe the same point on the unit circle.

In [ ]:
arguments = [QQ(1) / 4, QQ(5) / 4, -QQ(3) / 4]
values = [trefoil_signature(argument) for argument in arguments]
list(zip(arguments, values))

Use Sage rationals such as `QQ(1)/4` for mathematically distinguished points. Exact keys allow contributions computed by different summands to cancel exactly.

## 4. Constructing and combining sparse functions

A `SignatureFunction` may also be constructed directly from `(location, weight)` pairs. Repeated locations are added and zero totals are removed.

In [ ]:
toy = SignatureFunction(
    values=[
        (QQ(1) / 4, -1),
        (QQ(1) / 4, -2),
        (QQ(3) / 4, 3),
    ],
    plot_title="toy function",
)

print(toy.jumps_counter)
print("balanced:", toy.total_sign_jump() == 0)

The usual pointwise algebra acts directly on weights. A concordance inverse negates a knot signature, so a knot added to its inverse has no remaining jumps.

In [ ]:
cancelled = trefoil_signature + (-trefoil_signature)
scaled_and_shifted = (2 * trefoil_signature) >> (QQ(1) / 12)

print("trefoil plus its negative is zero:", cancelled.is_zero_everywhere())
print("shifted jump data:", scaled_and_shifted.jumps_counter)

`>> r` rotates jump locations forward by `r` modulo one; `<< r` rotates them backward. Scalar multiplication changes the signature values but not the jump locations. Every operation creates a new object, leaving its operands unchanged.

## 5. The satellite formula in the stored representation

For a `(p,q)`-cable, the ordinary signature is the sum of the pattern signature and the companion signature evaluated at `omega^p`. In argument coordinates this sends `x` to `p*x`. For `p=2`, `double_cover()` constructs precisely that pullback.

In [ ]:
pattern = GeneralizedAlgebraicKnot.torus_knot(2, 5)
cable = GeneralizedAlgebraicKnot.iterated_torus_knot(
    [(2, 3), (2, 5)]
)

formula_signature = pattern.signature() + trefoil_signature.double_cover()
computed_signature = cable.signature()

print("satellite formula verified:", computed_signature == formula_signature)
print("number of nonzero jump locations:", len(computed_signature.jumps_counter))

This equality is stronger than comparing a few sample values: equality of `SignatureFunction` objects compares the complete sparse jump data.

## 6. Plotting

The historical public class name is `SignaturePloter` (with one `t`). Passing an existing Matplotlib axes and `subplot=True` prevents the helper from opening or saving a separate figure.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
SignaturePloter.plot(
    computed_signature,
    subplot=True,
    ax=ax,
    title="Levine--Tristram signature of T(2,3; 2,5)",
    ylabel="signature",
)
ax.set_xlabel("argument x in [0,1)")
fig.tight_layout()
fig

The horizontal segments show one-sided constant values. The exact value at a discontinuity is still governed by the midpoint convention demonstrated above.

## Exercises

1. Compute and plot the signature of `T(3,4)`.
2. Evaluate it immediately to the left, at, and immediately to the right of one stored jump.
3. Verify the satellite formula for the `(2,7)`-cable of the trefoil using `double_cover()`.
4. Form the signature of a connected sum and verify that it equals the sum of the component signature functions.